In [5]:
import numpy as np
import sentencepiece as spm
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences


In [6]:
# MUST match training
SEQ_LEN = 8         # change if you trained with 5 / 10
MODEL_PATH = "gru_autocomplete_sentencepiece_v2.h5"
SP_MODEL_PATH = "sentencepiece_autocomplete_v2.model"


In [7]:
sp = spm.SentencePieceProcessor()
sp.load(SP_MODEL_PATH)

VOCAB_SIZE = sp.get_piece_size()
print("SentencePiece vocab size:", VOCAB_SIZE)


SentencePiece vocab size: 24000


In [8]:
model = load_model(MODEL_PATH)
model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 8, 128)            3072000   
                                                                 
 gru (GRU)                   (None, 256)               296448    
                                                                 
 dense (Dense)               (None, 256)               65792     
                                                                 
 dense_1 (Dense)             (None, 24000)             6168000   
                                                                 
Total params: 9,602,240
Trainable params: 9,602,240
Non-trainable params: 0
_________________________________________________________________


In [9]:
STOP_PIECES = {"▁the", "▁a", "▁an", "▁of", "▁to", "▁in", "▁and"}
STOP_IDS = {sp.piece_to_id(p) for p in STOP_PIECES if sp.piece_to_id(p) != -1}

print("Blocked stopword IDs:", STOP_IDS)


Blocked stopword IDs: {32, 3, 9, 50, 20, 23, 24}


In [10]:
def predict_next(text):
    ids = sp.encode(text, out_type=int)

    if not ids:
        return ""

    ids = ids[-SEQ_LEN:]
    ids = pad_sequences([ids], maxlen=SEQ_LEN)

    preds = model.predict(ids, verbose=0)[0]

    # block stopwords
    for sid in STOP_IDS:
        preds[sid] = 0.0

    next_id = int(np.argmax(preds))
    piece = sp.id_to_piece(next_id)

    # make it human-readable
    return piece.replace("▁", " ").strip()


In [12]:
tests = [
    "Dr.Trump is our project",
    "Paris is capital of",
]

for t in tests:
    print(f"{t:<35} → {predict_next(t)}")

Dr.Trump is our project             → "
Paris is capital of                 → its
